# 01 · Prompt와 모델

**목표**: Gen AI Hub SDK로 LLM을 호출하고, message role과 system instruction이 출력에 미치는 영향을 비교합니다.

LLM은 다음 토큰을 예측하는 모델이며, 우리가 보내는 입력 전체를 prompt라고 합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Prompt Chatbot (UI)</h2>
<p style="margin: 7px 0;">아래 UI에서 system prompt, 모델, 실행 방식을 바꾸며 대화를 이어가세요. ReAct Agent는 필요할 때 읽기 전용 계산 도구를 호출한 뒤, 그 관찰값을 답변에 반영합니다. 대화 history는 현재 브라우저 session에만 유지됩니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;">
<li>질문을 입력하고 답변을 확인합니다.</li>
<li>⚙️ 설정에서 일반 Chat 또는 ReAct Agent를 선택합니다. ReAct Agent에서는 <code>15% 할인 뒤 3개 가격은?</code>처럼 계산이 필요한 질문을 입력해 보세요.</li>
<li>이 UI는 업무 상태를 변경하지 않습니다.</li>
</ol>
  </div>
</div>


In [1]:
from pathlib import Path
import json
from time import perf_counter
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '2_gen_ai_hub_llm')]
from src.prompt_model.lab_core import run_adaptive_react_agent, run_chat, run_chat_stream, run_react_agent
from src.notebook_ui import build_chat_ui
chat_ui = build_chat_ui(run_chat, chat_stream_runner=run_chat_stream, react_agent_runner=run_react_agent, adaptive_react_agent_runner=run_adaptive_react_agent)
chat_ui.launch(inline=True, share=False, server_name='127.0.0.1', show_error=True, width='100%', height=720)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">메시지 role과 준비</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li><code>system</code>은 모델의 역할, 규칙, 언어, 출력 형식을 정합니다.</li>
<li><code>user</code>는 사용자의 질문 또는 요청입니다.</li>
<li><code>assistant</code>는 이전 모델 응답으로, 대화 history에 사용합니다.</li>
</ul>
<p style="margin: 7px 0;">아래 코드는 루트 <code>.env</code>의 <code>LLM_MODEL</code>을 사용합니다. UI를 실행한 뒤에는 선택 심화 자료로 코드 기반 비교를 확인하세요.</p>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


In [21]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '2_gen_ai_hub_llm')]
from src.prompt_model.lesson_support.llm import chat_text
from src.prompt_model.lesson_support.config import load_settings
from src.prompt_model.lab_core import compare_prompts, run_chat
settings = load_settings()
print(f'LLM model: {settings.llm_model}')

LLM model: gpt-5.6-luna


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 prompt 비교하기</h2>
<p style="margin: 7px 0;">이 아래 셀들은 UI와 같은 LLM 호출을 공개 코어 <code>compare_prompts()</code>로 확인하는 심화 자료입니다. 기본 실습에는 실행할 필요가 없습니다.</p>
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">기본 chat 응답</h3>
<p style="margin: 7px 0;">같은 질문을 기본 조건과 표 형식 system 조건으로 함께 호출하고, 먼저 기본 응답을 확인합니다.</p>
  </div>
</div>


In [13]:
question = 'LLM의 기본 아키텍처를 설명해줘.'
comparison_system_prompt = """너는 친절한 선생님이다.
초등학생도 이해할 수 있게 쉬운 한국어로 설명한다.
첫 줄에는 한 문장 요약을 쓴다.
그 아래에 핵심 내용을 bullet 3개로 정리한다."""

# comparison_system_prompt를 직접 바꿔 원하는 제약을 실험하세요.
comparison = compare_prompts(question, settings, system_prompt=comparison_system_prompt)
print(comparison['baseline'])

LLM(Large Language Model)의 기본 아키텍처는 대부분 **Transformer**를 기반으로 합니다. 대표적으로 GPT 계열은 Transformer의 **디코더(decoder) 구조**를 사용합니다.

## 1. 입력 토큰화(Tokenization)

문장을 모델이 처리할 수 있는 작은 단위인 **토큰(token)** 으로 나눕니다.

예를 들어:

```text
"나는 학교에 간다"
→ ["나는", "학교", "에", "간다"]
```

실제로는 단어 전체가 아니라 단어 조각이나 문자 단위로 나뉠 수 있습니다.

각 토큰은 고유한 정수 ID로 변환됩니다.

---

## 2. 임베딩(Embedding)

토큰 ID를 고정된 길이의 실수 벡터로 변환합니다.

```text
토큰 ID → 벡터
```

이 벡터에는 토큰의 의미와 다른 토큰과의 관계가 학습됩니다.

예를 들어 “왕”과 “여왕”은 서로 관련된 방향의 벡터로 표현될 수 있습니다.

LLM은 보통 다음 정보를 함께 사용합니다.

- **토큰 임베딩**: 토큰 자체의 의미
- **위치 정보**: 문장에서 토큰이 등장한 순서

위치 정보가 필요한 이유는 다음 두 문장이 서로 다른 의미를 가질 수 있기 때문입니다.

```text
고양이가 개를 쫓는다
개가 고양이를 쫓는다
```

---

## 3. Transformer 블록

LLM의 핵심은 여러 개의 **Transformer 블록**을 층층이 쌓은 구조입니다. 각 블록은 주로 다음 요소로 구성됩니다.

### 3.1 자기 주의 메커니즘(Self-Attention)

각 토큰이 문장 안의 다른 토큰을 얼마나 참고해야 하는지 계산합니다.

예를 들어:

```text
철수가 영희에게 책을 주었다. 그는 기뻐했다.
```

여기서 “그”가 누구를 가리키는지 이해하려면 앞의 토큰들과의 관계를 살펴봐야 합니다. Self-Attention은 이러한 관계를 계산합니다.

각 토큰은 세 가지 벡터로 변환됩니다.

- Query: 무엇을 찾는가
- K

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">system prompt로 제약하기</h3>
<p style="margin: 7px 0;">같은 코어 호출에서 반환된 표 형식 결과를 확인합니다. 제목·표·bullet처럼 검증하기 쉬운 제약은 기본 응답과 차이를 더 분명하게 만듭니다.</p>
  </div>
</div>


In [14]:
print(comparison['constrained'])

LLM은 글을 작은 조각으로 바꾸고, 문맥을 이해해 다음에 올 단어를 예측하는 인공지능입니다.

- **입력 변환**: 문장을 토큰이라는 작은 단위로 나누고, 각 토큰을 숫자 벡터인 임베딩으로 바꿉니다. 문장 속 위치 정보도 함께 넣습니다.
- **문맥 이해**: 트랜스포머의 **어텐션** 기능이 문장 안의 단어들을 서로 비교해 중요한 관계를 찾습니다. 여러 층을 지나며 문맥과 의미를 점점 더 깊게 이해합니다.
- **다음 토큰 예측**: 마지막에는 지금까지의 내용을 바탕으로 다음에 올 토큰들의 확률을 계산합니다. 가장 알맞은 토큰을 하나씩 이어 붙여 답변을 만듭니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 2</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">message 배열을 직접 구성하기</h2>
<p style="margin: 7px 0;">LLM API에는 하나의 문자열이 아니라 역할이 붙은 메시지 배열을 보냅니다. 아래에서 <code>system</code> 규칙, 이전 <code>user</code> 질문, 이전 <code>assistant</code> 답변, 새 <code>user</code> 질문의 순서를 확인하고 실행하세요.</p>
  </div>
</div>


In [15]:
messages = [
    {"role": "system", "content": "너는 친절한 교육 보조자다. 한국어로 간결하게 답한다."},
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
    {"role": "user", "content": "방금 답을 표로 정리해줘."},
]
for index, item in enumerate(messages, start=1):
    print(f'{index}. {item["role"]}: {item["content"]}')

reply = chat_text(settings, messages, settings.llm_model)
print()
print('응답:')
print(reply)

1. system: 너는 친절한 교육 보조자다. 한국어로 간결하게 답한다.
2. user: LLM의 장점을 두 가지 알려줘.
3. assistant: 자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다.
4. user: 방금 답을 표로 정리해줘.

응답:
| 장점 | 설명 |
|---|---|
| 자연어 이해 | 사람의 질문과 문장을 이해하고 자연스럽게 답변할 수 있습니다. |
| 다양한 작업 지원 | 요약, 번역, 글쓰기, 아이디어 생성 등을 도울 수 있습니다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 3</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">대화 history가 맥락을 전달하는 방식</h2>
<p style="margin: 7px 0;">두 번째 질문만 보면 무엇을 표로 바꾸라는지 알 수 없습니다. 같은 질문을 history 없이, 그리고 history와 함께 보내 결과를 비교해 보세요.</p>
<p style="margin: 7px 0;"><strong>관찰:</strong> 코어는 최근 12개 메시지와 총 12,000자 안에서만 history를 보존합니다. 긴 대화에서는 중요한 맥락을 요약하거나 문서 검색으로 보완해야 합니다.</p>
  </div>
</div>


In [16]:
follow_up = '방금 답을 표로 정리해줘.'
history = [
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
]
without_history = run_chat(follow_up, settings)
with_history = run_chat(follow_up, settings, history=history)
print('[history 없음]')
print(without_history['reply'])
print()
print('[history 포함]')
print(with_history['reply'])

[history 없음]
정리할 **이전 답변 내용이 현재 대화에 보이지 않습니다.**  
표로 만들 답변을 붙여 넣어 주시면 아래처럼 깔끔하게 정리해 드리겠습니다.

| 항목 | 내용 |
|---|---|
| 핵심 내용 |  |
| 주요 특징 |  |
| 장점 |  |
| 단점/주의사항 |  |
| 결론 |  |

[history 포함]
| 번호 | LLM의 장점 |
|---|---|
| 1 | 자연어를 이해할 수 있습니다. |
| 2 | 다양한 텍스트 작업을 도울 수 있습니다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 4</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">프롬프트로 JSON 형식 요청하고 검증하기</h2>
<p style="margin: 7px 0;">형식을 요청해도 항상 파싱 가능한 JSON이 반환된다고 보장할 수는 없습니다. 실행 결과가 파싱에 성공하는지 확인하고, 실패했다면 모델이 반환한 원문을 살펴보세요.</p>
  </div>
</div>


In [17]:
json_prompt = (
    '너는 교육 보조자다. 반드시 JSON 객체만 반환한다. Markdown 코드 블록이나 설명을 덧붙이지 않는다. '
    '키는 summary(문자열), keywords(문자열 배열)만 사용한다.'
)
json_outcome = run_chat(
    'LLM을 한 문장으로 요약하고 핵심 키워드 세 개를 추출해줘.',
    settings,
    system_prompt=json_prompt,
)
try:
    parsed = json.loads(json_outcome['reply'])
    print('JSON 파싱 성공:', parsed)
except json.JSONDecodeError:
    print('JSON 파싱 실패. 반환 원문을 확인하세요:')
    print(json_outcome['reply'])

JSON 파싱 성공: {'summary': 'LLM은 방대한 텍스트 데이터를 학습해 인간과 유사한 언어를 이해하고 생성하는 인공지능 모델이다.', 'keywords': ['대규모 언어 모델', '자연어 처리', '생성형 인공지능']}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 실험</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">동일 질문을 모델별로 비교하기</h2>
<p style="margin: 7px 0;">현재 모델과 AI Core에 배포된 다른 모델을 같은 질문으로 호출해 보세요. 응답 시간뿐 아니라 정확성, 지시 준수, 출력 형식을 함께 기록합니다. 배포되지 않은 모델명은 호출할 수 없습니다.</p>
  </div>
</div>


In [18]:
models_to_compare = [settings.llm_model]
# AI Core에 RUNNING 상태로 배포된 모델이 있다면 아래처럼 추가합니다.
models_to_compare.append('gpt-4.1')

comparison_question = 'SAP에서 2026년도에 출시한 제품은?'
for model_name in models_to_compare:
    started_at = perf_counter()
    outcome = run_chat(comparison_question, settings, model=model_name)
    elapsed = perf_counter() - started_at
    print(f'[{model_name}] {elapsed:.1f}초')
    print(outcome['reply'])
    print('='*100 + '\n')

[gpt-5.6-luna] 7.2초
“2026년에 출시한 SAP 제품”은 **신규 제품**인지, 기존 제품의 **2026년 버전·릴리스**인지에 따라 달라집니다. SAP는 매년 전체 제품 출시 목록을 하나로 공개하지 않고, 제품별로 릴리스합니다.

현재 기준으로는 다음을 구분해야 합니다.

- **신규 제품 출시**: SAP가 2026년에 처음 발표·판매한 제품
- **버전 출시**: SAP S/4HANA Cloud, SAP SuccessFactors, SAP Ariba 등 기존 제품의 2026년 릴리스
- **지역별 출시**: 한국에서 판매·서비스가 시작된 제품
- **기능 출시**: Joule, SAP Business AI 등의 신규 기능이나 서비스

따라서 정확한 목록을 만들려면 **글로벌 기준인지, 한국 출시 기준인지**, 그리고 **신규 제품만 볼지 2026년 릴리스까지 포함할지**가 필요합니다. SAP 공식 발표는 [SAP News Center](https://news.sap.com/)와 각 제품의 **What’s New Viewer**에서 확인하는 것이 가장 정확합니다.

[gpt-4.1] 2.8초
현재까지의 정보(2024년 6월 기준)로는 SAP에서 2026년도에 출시 예정이거나 출시된 제품에 대한 공식 발표가 없습니다. SAP는 주로 새로운 클라우드 솔루션, S/4HANA, 인공지능(AI) 통합 기능, 비즈니스 네트워크 등의 신제품 및 기능 개선을 지속적으로 발표하고 있습니다. 그러나 2026년에 특정 제품이 출시된다는 공식 뉴스나 보도자료는 아직 공개되지 않았습니다.

만약 SAP의 2026년 제품 출시 정보가 필요하다면 공식 SAP 홈페이지, 뉴스룸, 또는 공식 보도자료를 참고하는 것이 가장 확실합니다. 현재로서는 2026년에 어떤 제품이 출시될지는 공개되지 않았거나 정보가 없습니다.



<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #7b1fa2; background: #f8f1fb; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #7b1fa2; letter-spacing: .02em;">🤖  ReAct 코드 실습</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">ReAct Agent의 답변 생성 과정</h2>
<p style="margin: 7px 0;">계산 도구 실행 뒤 Agent가 최종 답변을 만드는 과정을 확인합니다. <code>Thinking</code>은 상세 내부 추론이 아니라 공개 가능한 요약만 표시합니다.</p>
  </div>
</div>


In [19]:
react_question = '정가 12,000원인 상품을 15% 할인해 3개 구매하면 총 얼마인가요?'
react_outcome = run_react_agent(
    message=react_question,
    settings=settings,
    system_prompt='계산 결과와 계산 근거를 한국어로 간결하게 설명한다.',
)

print('[ReAct 실행 기록]')
for index, step in enumerate(react_outcome['trace'], start=1):
    print(f'\nStep {index}')
    print('Thinking (공개 추론 요약):', step.get('thinking', '-'))
    print('Action:', step.get('action', '-'))
    print('Observation:', step.get('observation', '-'))
    print('Evaluation:', step.get('evaluation', '-'))

print('\n[최종 답변]')
print(react_outcome['reply'])

print('\n[태그 응답 · 후속 파싱용]')
print(react_outcome['structured_reply'])


[ReAct 실행 기록]

Step 1
Thinking (공개 추론 요약): 할인된 개당 가격을 구한 뒤 3개를 곱해야 하므로, 식 `12000 × (1 - 0.15) × 3`을 계산합니다.
Action: calculate({"expression": "12000*(1-0.15)*3"})
Observation: {"expression": "12000*(1-0.15)*3", "value": 30600.0}
Evaluation: 도구 결과를 받았습니다. 이 관찰값을 다음 응답 생성에 사용합니다.

Step 2
Thinking (공개 추론 요약): 도구 관찰값을 바탕으로 사용자에게 전달할 답을 정리합니다.
Action: 최종 답변 생성
Observation: 15% 할인 후 개당 가격은 10,200원입니다.

3개 총액: 10,200원 × 3 = **30,600원**
Evaluation: 도구 실행 결과를 사용한 최종 응답 단계를 마쳤습니다.

[최종 답변]
15% 할인 후 개당 가격은 10,200원입니다.

3개 총액: 10,200원 × 3 = **30,600원**

[태그 응답 · 후속 파싱용]
<thinking>도구 관찰값을 바탕으로 사용자에게 전달할 답을 정리합니다.</thinking>
<answer>15% 할인 후 개당 가격은 10,200원입니다.

3개 총액: 10,200원 × 3 = **30,600원**</answer>
<evaluation>도구 실행 결과를 사용한 최종 응답 단계를 마쳤습니다.</evaluation>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 미션</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">사내 FAQ 답변 도우미의 system prompt 설계</h2>
<p style="margin: 7px 0;">아래 셀에서 대상 독자, 말투, 답변 형식, 근거가 부족할 때의 행동을 포함한 system prompt를 작성하세요. 같은 질문을 기본 응답과 비교하고, 요구한 형식과 안전 규칙이 지켜졌는지 평가합니다.</p>
  </div>
</div>


In [20]:
faq_system_prompt = '''
TODO: 사내 FAQ 답변 도우미의 규칙을 작성하세요.
'''
faq_question = '출장비 정산 마감일이 언제인가요?'

# 작성 후 아래 주석을 해제해 기본 응답과 비교하세요.
# print(run_chat(faq_question, settings)['reply'])
# print(run_chat(faq_question, settings, system_prompt=faq_system_prompt)['reply'])